# 03 · Knowledge Graph (Người 3, slide 17–23)

Dựng đồ thị bất động sản theo schema, nạp dữ liệu, kiểm tra nhất quán, truy vấn bằng Cypher và chạy thuật toán đồ thị.

**Đầu ra cho người sau:** đồ thị Neo4j, `graph/load.cypher`, `src/graph.py`: `run_cypher`, `ask_graph`.

**Cần chạy trước:** `docker compose up -d graph_db`.

**Lưu ý:** P3-12 chạy Leiden trên đồ thị tương đồng giữa các phường, không chạy trên cây hành chính. P3-10 là mẫu Text2Cypher cho P4-03. P3-08 là cell tuỳ chọn.

In [ ]:
%run ./00_setup.ipynb

### P3-01 · Kết nối Neo4j, đồ thị dựng từ dữ liệu sạch của Người 1
**ẨN** · slide 17 · Người 3

Mục tiêu: kết nối Neo4j (docker compose up -d graph_db) và nạp dữ liệu sạch.

In [ ]:
# P3-01 · ẨN
df = pd.read_parquet(CLEAN_PATH)
graph.driver()  # lỗi ở đây: chạy `docker compose up -d graph_db` rồi thử lại

### P3-02 · Một tin đăng là 6 nút và 5 cạnh
**LIVE** · slide 17 · Người 3

Mục tiêu: đồ thị mini 6 nút dựng tay từ 1 tin thật; in danh sách triple.

In [ ]:
# P3-02 · LIVE
row = {**df.set_index("listing_id").loc[1].to_dict(), "listing_id": 1}
G, triples = graph.mini_graph(row)
for t in triples:
    print(t)
assert G.number_of_nodes() == 6 and len(triples) == 5
graph.draw_graph(G, title=f"Tin #1: {G.number_of_nodes()} nút, {G.number_of_edges()} cạnh");

### P3-03 · Schema 5 nhãn nút, mỗi nhãn có ràng buộc khoá duy nhất
**CACHE** · slide 19 · Người 3

Mục tiêu: tạo ràng buộc và chỉ mục theo schema (Listing, PropertyType, Ward, District, Province).

In [ ]:
# P3-03 · CACHE
graph.run_block("constraints")
constraints = run_cypher("SHOW CONSTRAINTS YIELD name, labelsOrTypes, properties")
print(constraints.to_string(index=False))
graph.draw_graph(graph.schema_graph(), title="Schema đồ thị bất động sản");

### P3-04 · Chuẩn hoá địa danh: "Q.7" thành "Quận 7"
**LIVE** · slide 20 · Người 3

Mục tiêu: entity resolution: hàm chuẩn hoá địa danh; khoá Ward gồm cả quận và tỉnh.

In [ ]:
# P3-04 · LIVE
samples = [("district", "Q.7"), ("district", "q7"), ("district", "8"), ("district", "Huyện Củ Chi"),
           ("ward", "P.5"), ("ward", "Phường 05"), ("province", "TP. Hồ Chí Minh"), ("province", "Tp Hà Nội")]
table = pd.DataFrame([{"loại": k, "trước": v, "sau": data.normalize_place(v, k)} for k, v in samples])
assert data.normalize_place("Q.7", "district") == "Quận 7"
same_name = df.dropna(subset=["ward"]).groupby("ward")["district"].nunique()
w = same_name.idxmax()
print(f'"{w}" xuất hiện ở {same_name.max()} quận khác nhau, nên khoá Ward = "tỉnh|quận|phường"')
table

### P3-05 · Nạp node địa danh và loại hình bằng UNWIND theo lô
**CACHE** · slide 20 · Người 3

Mục tiêu: nạp node Province, District, Ward, PropertyType bằng UNWIND theo lô (graph/load.cypher).

In [ ]:
# P3-05 · CACHE
rows = graph.graph_rows(df)
if REBUILD or not graph.graph_matches(df):  # dữ liệu đổi: xoá đồ thị cũ rồi nạp lại
    graph.reset_graph()
    for name in ("provinces", "districts", "wards", "property_types"):
        graph.run_block(name, rows[name])
run_cypher("MATCH (n) WHERE NOT n:Listing RETURN labels(n)[0] AS loai, count(*) AS so_node ORDER BY loai")

### P3-06 · Đồ thị có [ĐIỀN SAU] node và [ĐIỀN SAU] edge
**CACHE** · slide 20, 23 · Người 3

Mục tiêu: nạp Listing và các quan hệ OF_TYPE, IN_WARD, IN_DISTRICT, IN_PROVINCE.

In [ ]:
# P3-06 · CACHE
if REBUILD or not graph.graph_matches(df):
    with Timer() as t:
        for name in ("listings", "of_type", "in_ward", "in_district", "in_province"):
            graph.run_block(name, rows[name])
    print(f"nạp xong trong {t.seconds:.0f} s")
counts = graph.graph_counts()
print("node:", counts.query("nhom == 'node'").so_luong.sum(), "| edge:", counts.query("nhom == 'edge'").so_luong.sum())
counts

### P3-07 · Số node và edge trong đồ thị khớp với pandas
**CACHE** · slide 23 · Người 3

Mục tiêu: kiểm tra nhất quán: số node, edge trong đồ thị so với pandas (assert).

In [ ]:
# P3-07 · CACHE
check = graph.expected_counts(df).merge(graph.graph_counts(), on="loai", how="left")
assert (check["pandas"] == check["so_luong"]).all(), check
check[["loai", "pandas", "so_luong"]]

### P3-08 · LLM trích được tiện ích từ mô tả, đúng [ĐIỀN SAU]%
**CACHE** · slide 20 · Người 3

Mục tiêu: tuỳ chọn: LLM trích tiện ích trên 20 tin mẫu, kiểm tra tay.

In [ ]:
# P3-08 · CACHE
sample = df.dropna(subset=["description"]).sample(20, random_state=SEED)
amen = cached("p3_08_amenities.parquet", lambda: pd.DataFrame(
    [{"listing_id": r.listing_id, "tiện ích": a["name"], "bằng chứng": a["evidence"]}
     for r in sample.itertuples() for a in graph.extract_amenities(r.description)]))
text = sample.set_index("listing_id")["description"].str.lower()
amen["nguyên văn"] = [e.lower() in text[i] for i, e in zip(amen.listing_id, amen["bằng chứng"])]
amen.to_csv(CACHE_DIR / "p3_08_kiem_tra_tay.csv", index=False)  # điền cột 'đúng' bằng tay
print(f"{len(amen)} triple từ 20 tin | bằng chứng khớp nguyên văn: {amen['nguyên văn'].mean():.0%}")
show(amen, rows=8)

### P3-09 · Cypher tra cứu đọc như câu hỏi, tương đương một câu SQL
**LIVE** · slide 21 · Người 3

Mục tiêu: Cypher 1, tra cứu: căn hộ 2PN dưới mức giá cho trước; in SQL tương đương bên cạnh.

In [ ]:
# P3-09 · LIVE
print(graph.QUERIES["lookup"].strip(), "\n\n-- SQL tương đương:\n" + graph.SQL_EQUIV["lookup"])
run_cypher(graph.QUERIES["lookup"], {"district": "Thủ Đức", "max_price": 3e9})

### P3-10 · Cypher tổng hợp khớp pandas groupby
**LIVE** · slide 21 · Người 3

Mục tiêu: Cypher 2, tổng hợp: quận có giá/m² trung bình thấp nhất; assert bằng kết quả pandas groupby.

In [ ]:
# P3-10 · LIVE
cy = run_cypher(graph.QUERIES["aggregate"], {"province": "Hồ Chí Minh", "min_n": 20})
g = (df[df.province == "Hồ Chí Minh"].dropna(subset=["district", "price_m2_mil"])
     .groupby("district")["price_m2_mil"].agg(["mean", "count"]).query("count >= 20"))
assert cy.district[0] == g["mean"].idxmin() and abs(cy.avg_price_m2[0] - g["mean"].min()) < 0.01
print(f"Cypher: {cy.district[0]} {cy.avg_price_m2[0]} tr/m² ({cy.n[0]} tin)")
print(f"pandas: {g['mean'].idxmin()} {g['mean'].min():.2f} tr/m² ({g['count'][g['mean'].idxmin()]} tin)")

### P3-11 · Đi bốn chặng Tin → Phường → Quận → Tỉnh để đếm tin theo quận
**LIVE** · slide 21 · Người 3

Mục tiêu: Cypher 3, đa chặng: đếm tin theo quận qua đường Tin, Phường, Quận, Tỉnh.

In [ ]:
# P3-11 · LIVE
top5 = run_cypher(graph.QUERIES["multihop"], {"province": "Hồ Chí Minh"})
print(graph.QUERIES["multihop"].strip())
# Đường đi bắt buộc qua Phường: tin thiếu phường không được đếm, khác với đếm theo cột quận.
by_col = df[df.province == "Hồ Chí Minh"].district.value_counts()
top5.assign(theo_cot_quan=top5.district.map(by_col))

### P3-12 · Modularity Q = 0,357 ở toy; Leiden gom phường có hồ sơ giá giống nhau
**LIVE** · slide 22 · Người 3

Mục tiêu: tính Q toy và assert bằng 5/14; dựng đồ thị tương đồng giữa các phường (kNN) rồi chạy Leiden.

In [ ]:
# P3-12 · LIVE
edges = [(0, 1), (1, 2), (0, 2), (3, 4), (4, 5), (3, 5), (2, 3)]
Q = graph.modularity(edges, [{0, 1, 2}, {3, 4, 5}])
assert abs(Q - 5 / 14) < 1e-9
centrality = {v: sum(v in e for e in edges) / 5 for v in range(6)}
assert centrality[2] == centrality[3] == 0.6 and centrality[0] == 0.4

prof, Q_real = graph.ward_communities(graph.ward_profiles(df), k=5, seed=SEED)
print(f"toy Q = {Q:.3f} | Leiden trên {len(prof)} phường (kNN, k = 5): "
      f"{prof.community.nunique()} cộng đồng, Q = {Q_real:.3f}")
graph.community_table(prof)

### P3-13 · Một quận nhìn như một chùm sao quanh các phường
**CACHE** · slide 23 · Người 3

Mục tiêu: trực quan hoá một quận (tối đa 50 nút) bằng pyvis.

In [ ]:
# P3-13 · CACHE
from IPython.display import IFrame
path = CACHE_DIR / "p3_13_cau_giay.html"
if REBUILD or not path.exists():
    graph.district_html("Cầu Giấy", "Hà Nội", path, max_nodes=50)
IFrame(os.path.relpath(path), width="100%", height=500)